In [1]:
%pip install -r ../../requirements.txt


     -------------------------------------- 160.4/160.4 kB 1.4 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
     ---------------------------------------- 93.8/93.8 kB 2.7 MB/s eta 0:00:00
     ---------------------------------------- 78.4/78.4 kB 2.2 MB/s eta 0:00:00
     --------------------------------------- 14.6/14.6 MB 19.9 MB/s eta 0:00:00
     -------------------------------------- 261.5/261.5 kB 8.1 MB/s eta 0:00:00
     ---------------------------------------- 139.8/139.8 kB ? eta 0:00:00
     --------------------------------------- 12.4/12.4 MB 24.2 MB/s eta 0:00:00
     ---------------------------------------- 2.2/2.2 MB 28.0 MB/s eta 0:00:00
     ------------------------------------- 914.9/914.9 kB 19.2 MB/s eta 0:00:00
     ---------------------------------------- 76.7/76.7 kB ? eta 0:00:00
     ------------------------------------- 388.2/388.2 kB 11.8 MB/s eta 0:00:00
     --------------------------

  DEPRECATION: liac-arff is being installed using the legacy 'setup.py install' method, because it does not have a 'pyproject.toml' and the 'wheel' package is not installed. pip 23.1 will enforce this behaviour change. A possible replacement is to enable the '--use-pep517' option. Discussion can be found at https://github.com/pypa/pip/issues/8559

[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


<a target="_blank" href="https://colab.research.google.com/github/cesarschoollectures/am-labs/blob/main/assignments/E01_Decision_Tree.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta atividade, você irá trabalhar com o dataset Fashion MNIST utilizando modelos de classificação do sklearn.

O foco NÃO é apenas obter bons resultados, mas garantir que o experimento seja:
- correto
- reprodutível
- bem estruturado
- criticamente analisado

# Dicas importantes

## Sobre o dataset (Fashion MNIST)

- Utilize `fetch_openml` do sklearn para carregar os dados
- Use: `as_frame=False`
- Use: `mnist_784`
- Converta os rótulos para inteiro:
  
  ```python
  y = y.astype(int)
  ```

# Questão 1

Implemente uma função load_data(seed) que:

Carregue o dataset `Fashion MNIST`
Realize a separação em treino e teste
Utilize `train_test_split` com controle de aleatoriedade
Retorne: `X_train`, `X_test`, `y_train`, `y_test`

Depois responda: 
É necessário normalizar os dados para esse tipo de modelo? Justifique.

**Solução**:

In [1]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.metrics import accuracy_score, classification_report

def load_data(seed=42):
    mnist = fetch_openml('mnist_784', as_frame=False)
    X, y = mnist.data, mnist.target.astype(int)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=seed
    )
    return X_train, X_test, y_train, y_test


Não, o random forest e adaBoost são modelos baseados em árvore de decisão que tem como caraterística não necessitar de normalização/tratamento dos dados.


# Questão 2

Implemente as funções:

`train_random_forest(X_train, y_train, seed)`
`train_adaboost(X_train, y_train, seed)`

## Requisitos:

Utilizar os modelos do `sklearn`
Garantir reprodutibilidade com `random_state`

**Solução**:

In [2]:
def train_random_forest(X_train, y_train, seed=42):
    model = RandomForestClassifier(n_estimators=100, random_state=seed)
    model.fit(X_train, y_train)
    return model

def train_adaboost(X_train, y_train, seed=42):
    model = AdaBoostClassifier(n_estimators=100, random_state=seed)
    model.fit(X_train, y_train)
    return model


# Questão 3

Implemente a função:

- `evaluate(model, X_test, y_test)`

Ela deve:
- Realizar predições
- Retornar a acurácia do modelo

**Solução**:

In [3]:
def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)
    return accuracy_score(y_test, y_pred)


# Questão 4

Implemente a função:

- `run_pipeline(model_type="rf", seed=42)`

Ela deve:
- Carregar os dados
- Treinar o modelo escolhido (`rf` ou `ab`)
- Avaliar o modelo
- Retornar a acurácia

**Solução**:

In [5]:
def run_pipeline(model_type="rf", seed=42):
    X_train, X_test, y_train, y_test = load_data(seed=seed)
    if model_type == "rf":
        model = train_random_forest(X_train, y_train, seed=seed)
    elif model_type == "ab":
        model = train_adaboost(X_train, y_train, seed=seed)
    else:
        raise ValueError(f"Tipo de modelo desconhecido: {model_type}")
    return evaluate(model, X_test, y_test)


In [6]:
from sklearn.tree import DecisionTreeClassifier

X_train, X_test, y_train, y_test = load_data(seed=42)

print(f"{'max_depth':<12} {'Treino':>8} {'Teste':>8} {'Gap':>8}")
print("-" * 40)
for depth in [1, 2, 3, 5, 8, 10, 15, 20, 30, None]:
    dt = DecisionTreeClassifier(max_depth=depth, random_state=42)
    dt.fit(X_train, y_train)
    train_acc = accuracy_score(y_train, dt.predict(X_train))
    test_acc  = accuracy_score(y_test,  dt.predict(X_test))
    label = str(depth) if depth is not None else "None"
    print(f"{label:<12} {train_acc:>8.4f} {test_acc:>8.4f} {train_acc - test_acc:>8.4f}")


max_depth      Treino    Teste      Gap
----------------------------------------
1              0.1968   0.2064  -0.0095
2              0.3430   0.3379   0.0052
3              0.4398   0.4447  -0.0049
5              0.6565   0.6577  -0.0012
8              0.8264   0.8094   0.0169
10             0.9049   0.8583   0.0466
15             0.9857   0.8742   0.1115
20             0.9952   0.8719   0.1232
30             0.9982   0.8733   0.1250
None           1.0000   0.8695   0.1305


**Em qual profundidade começa o overfitting?**

O overfitting começa a partir da profundidade 10. 

**Por que a árvore consegue 100% no treino quando max_depth=None?**

Quando isso acontece, a árvore de decisão cresce sem restrição de profundidade até que todas as folhas sejam puras. Com amostras suficientemente distintas, a árvore cria um caminho único para cada amostra de treino, decorando o conjunto de treino. Isso resulta em acurácia de treino de 100%, pois o modelo não generaliza padrões, mas decora as respostas, o que leva a baixa acurácia no teste quando as amostras são novas.


# Questão 5

Execute o pipeline para ambos os modelos:

- Random Forest
- AdaBoost

## Apresente:
- Acurácia, Precisão, Recall e F1-Score de cada modelo

## Responda:
- Qual modelo apresentou melhor desempenho inicial?

**Solução**:

In [ ]:
X_train, X_test, y_train, y_test = load_data(seed=42)

models = {
    "Random Forest": train_random_forest(X_train, y_train, seed=42),
    "AdaBoost":      train_adaboost(X_train, y_train, seed=42),
}

for name, model in models.items():
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"  {name}")
    print(f"Acurácia: {acc:.4f}")
    print(classification_report(y_test, y_pred))



  Random Forest
Acurácia: 0.9673
              precision    recall  f1-score   support

           0       0.98      0.99      0.99      1343
           1       0.98      0.98      0.98      1600
           2       0.95      0.97      0.96      1380
           3       0.96      0.95      0.96      1433
           4       0.96      0.97      0.97      1295
           5       0.97      0.96      0.97      1273
           6       0.98      0.98      0.98      1396
           7       0.97      0.97      0.97      1503
           8       0.96      0.95      0.96      1357
           9       0.96      0.95      0.95      1420

    accuracy                           0.97     14000
   macro avg       0.97      0.97      0.97     14000
weighted avg       0.97      0.97      0.97     14000


  AdaBoost
Acurácia: 0.7174
              precision    recall  f1-score   support

           0       0.90      0.63      0.74      1343
           1       0.90      0.90      0.90      1600
           2   

**Qual modelo apresentou melhor desempenho inicial?**

O Random Forest supera o AdaBoost, pois ele treina várias árvores profundas em paralelo usando bagging e seleção aleatória de features, capturando padrões complexos de forma robusta. O AdaBoost usa estimadores fracos de forma sequencial, o que é mais sensível a ruído e tende a ser mais lento para problemas de alta dimensão como imagens. Portanto, espera-se que o Random Forest apresente acurácia significativamente maior neste dataset.


# Questão 6

Execute o pipeline utilizando diferentes seeds (ex: 42 e 7).

## Analise:
- Os resultados mudaram?

## Responda:
- O experimento é reprodutível? Justifique.

**Solução**:

In [8]:
for seed in [42, 7]:
    acc_rf = run_pipeline("rf", seed=seed)
    acc_ab = run_pipeline("ab", seed=seed)
    print(f"Seed {seed:>2} | Random Forest: {acc_rf:.4f} | AdaBoost: {acc_ab:.4f}")


Seed 42 | Random Forest: 0.9673 | AdaBoost: 0.7174
Seed  7 | Random Forest: 0.9717 | AdaBoost: 0.7251


**O experimento é reprodutível? Justifique.**

Sim, a chave da reprodutibilidade está no parâmetro random_state passado a train_test_split, RandomForestClassifier e AdaBoostClassifier. Quando a mesma seed é usada, todas as operações (embaralhamento dos dados, sorteio de features, inicialização dos estimadores) produzem exatamente os mesmos resultados. A tabela acima confirma isso: rodar o pipeline duas vezes com seed=42 retorna valores idênticos. Mudar a seed pode alterar levemente as métricas por conta da diferente divisão treino/teste, mas cada combinação seed + modelo é sempre determinística.


# Questão 7

Para pelo menos um dos modelos:

- Compare a acurácia em treino e teste

## Responda:
- Existe overfitting?
- Qual modelo tende a sofrer mais com isso?

In [9]:
X_train, X_test, y_train, y_test = load_data(seed=42)

for name, model_fn in [("Random Forest", train_random_forest), ("AdaBoost", train_adaboost)]:
    model = model_fn(X_train, y_train, seed=42)
    train_acc = evaluate(model, X_train, y_train)
    test_acc  = evaluate(model, X_test, y_test)
    print(f"{name:<15} | Treino: {train_acc:.4f} | Teste: {test_acc:.4f} | Gap: {train_acc - test_acc:.4f}")


Random Forest   | Treino: 1.0000 | Teste: 0.9673 | Gap: 0.0327
AdaBoost        | Treino: 0.7246 | Teste: 0.7174 | Gap: 0.0072


**Existe overfitting? Qual modelo tende a sofrer mais com isso?**

O Random Forest com max_depth=None cresce árvores até folhas puras, portanto sua acurácia de treino é de 100%, logo, o overfitting é claro. No entanto, por conta do bagging e à seleção aleatória de features, o modelo generaliza razoavelmente bem no teste, e essa diferença treino-teste é controlada. O adaBoost, por usar estimadores fracos, tem menor overfitting individual por estimador.


# Questão 8

Varie pelo menos um hiperparâmetro em cada modelo:

- Random Forest: `n_estimators`
- AdaBoost: `n_estimators`

## Analise:
- O desempenho muda significativamente?

## Responda:
- Qual modelo é mais sensível a mudanças?

In [10]:
X_train, X_test, y_train, y_test = load_data(seed=42)

print("=== Random Forest — variando n_estimators ===")
for n in [10, 50, 100, 200]:
    model = RandomForestClassifier(n_estimators=n, random_state=42)
    model.fit(X_train, y_train)
    acc = evaluate(model, X_test, y_test)
    print(f"  n_estimators={n:>3}: {acc:.4f}")

print("\n=== AdaBoost — variando n_estimators ===")
for n in [10, 50, 100, 200]:
    model = AdaBoostClassifier(n_estimators=n, random_state=42)
    model.fit(X_train, y_train)
    acc = evaluate(model, X_test, y_test)
    print(f"  n_estimators={n:>3}: {acc:.4f}")


=== Random Forest — variando n_estimators ===
  n_estimators= 10: 0.9456
  n_estimators= 50: 0.9639
  n_estimators=100: 0.9673
  n_estimators=200: 0.9681

=== AdaBoost — variando n_estimators ===
  n_estimators= 10: 0.2428
  n_estimators= 50: 0.6426
  n_estimators=100: 0.7174
  n_estimators=200: 0.7676


**Qual modelo é mais sensível a mudanças em n_estimators?**

O adaBoost é mais sensível a variações em n_estimators. Com poucas iterações, o adaBoost ainda está "aprendendo" e apresenta desempenho bem inferior ao com 100 ou 200 estimadores, a diferença de acurácia entre 10 e 100 estimadores costuma ser maior para o adaBoost.


# Questão 9

Responda (máx. 2 parágrafos por item):

1. A acurácia é suficiente para avaliar os modelos?
2. Como você garante que o resultado não ocorreu por acaso?
3. Cite dois possíveis problemas metodológicos neste experimento.
4. O pipeline implementado é confiável? Justifique.

**1. A acurácia é suficiente para avaliar os modelos?**

Não necessariamente. A acurácia é uma métrica conveniente, mas insuficiente quando as classes são desbalanceadas, pois um classificador que sempre prediz a classe majoritária pode atingir alta acurácia sem ser útil. Para esse dataset (que possui classes balanceadas) a acurácia é razoável, mas ainda assim, métricas como precisão, recall e F1-Score por classe revelam padrões de erro mais ricos e devem complementar a análise.

**2. Como você garante que o resultado não ocorreu por acaso?**

Para garantir que os resultados são estatisticamente significativos deve-se utilizar validação cruzada, que avalia o modelo em múltiplas partições independentes dos dados e produz uma estimativa de variância.

**3. Dois possíveis problemas metodológicos neste experimento**

- **Avaliação em divisão única:** usar uma única divisão treino/teste pode gerar estimativas instáveis que dependem do sorteio aleatório específico. A validação cruzada estratificada forneceria estimativas mais robustas e com menor variância.
- **Seleção de hiperparâmetros no conjunto de teste:** se os hiperparâmetros forem escolhidos com base no desempenho observado no conjunto de teste ocorre *data leakage* indireto, tornando as estimativas de generalização otimistas. O correto seria usar um conjunto de validação separado.

**4. O pipeline implementado é confiável? Justifique.**

O pipeline é reprodutível graças ao uso de random_state controlado pela seed, garantindo que os experimentos possam ser replicados exatamente. No entanto, ele poderia ser mais confiável com o uso de validação cruzada em vez de uma única divisão treino/teste, e com uma etapa explícita de seleção de hiperparâmetros separada do conjunto de teste final. Para uso em produção, seriam recomendáveis ainda logging de métricas e verificações de consistência dos dados de entrada.
